In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Environment Setup, Secrets & W&B Initialization

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(seed=42)

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")
wandb.login(key=wandb_api_key)

wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="model1_tfidf_scratch_v4_pytorch",
    config={
        "model_type": "FROM_scratch_OPTION_SCORER_PYTORCH",
        "feature_extractor": "TFIDF_word+char+SVD",
        "word_tfidf_features": 4000,
        "char_tfidf_features": 3000,
        "svd_components": 150,
        "hidden_dim": 256,
        "dropout": 0.0,
        "weight_decay": 0.0001,
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 250,
        "n_folds": 5
    }
)

config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1000967 (23f1000967-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cuda


## Load Competition Data

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTS = ['A', 'B', 'C', 'D', 'E']
print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

Train Dataset Shape: (2000, 8)
Test Dataset Shape:  (500, 7)



## Text Feature Engineering Engine (Reading from config)

In [4]:
def build_corpus(df):
    corpus = df['prompt'].fillna("").tolist()
    for c in OPTS:
        corpus += df[c].fillna("").tolist()
    return corpus

full_corpus = build_corpus(train_df) + build_corpus(test_df)

word_vec = TfidfVectorizer(stop_words='english', max_features=config.word_tfidf_features,
                            sublinear_tf=True, ngram_range=(1, 2), min_df=2)
word_vec.fit(full_corpus)

char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                            max_features=config.char_tfidf_features, sublinear_tf=True, min_df=2)
char_vec.fit(full_corpus)

svd = TruncatedSVD(n_components=config.svd_components, random_state=42)
svd.fit(word_vec.transform(full_corpus))

TruncatedSVD(n_components=150, random_state=42)

In [5]:
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def build_features(df):
    n = len(df)
    feats = np.zeros((n, 5, 8), dtype=np.float32)

    prompts = df['prompt'].fillna("").tolist()
    opt_texts = {c: df[c].fillna("").tolist() for c in OPTS}

    p_word = word_vec.transform(prompts)
    p_char = char_vec.transform(prompts)
    p_svd = svd.transform(p_word)

    opt_word = {c: word_vec.transform(opt_texts[c]) for c in OPTS}
    opt_char = {c: char_vec.transform(opt_texts[c]) for c in OPTS}
    opt_svd = {c: svd.transform(opt_word[c]) for c in OPTS}

    for i in range(n):
        p_len = max(len(prompts[i]), 1)
        p_words = max(len(prompts[i].split()), 1)
        opt_lens = [len(opt_texts[c][i]) for c in OPTS]
        opt_word_counts = [len(opt_texts[c][i].split()) for c in OPTS]

        word_cos = np.array([cosine_similarity(p_word[i], opt_word[c][i])[0, 0] for c in OPTS])
        char_cos = [cosine_similarity(p_char[i], opt_char[c][i])[0, 0] for c in OPTS]
        svd_cos = [cosine_similarity(p_svd[i:i+1], opt_svd[c][i:i+1])[0, 0] for c in OPTS]
        jac = [jaccard(prompts[i], opt_texts[c][i]) for c in OPTS]
        len_ratio = [l / p_len for l in opt_lens]
        wc_ratio = [w / p_words for w in opt_word_counts]

        opt_vecs = [opt_word[c][i] for c in OPTS]
        centrality = [np.mean([cosine_similarity(opt_vecs[a], opt_vecs[b])[0, 0]
                                for b in range(5) if b != a]) for a in range(5)]

        word_cos_z = (word_cos - word_cos.mean()) / (word_cos.std() + 1e-6)

        for k in range(5):
            feats[i, k] = [word_cos[k], char_cos[k], svd_cos[k], jac[k],
                           len_ratio[k], wc_ratio[k], centrality[k], word_cos_z[k]]
    return feats

X_train_raw = build_features(train_df)
X_test_raw = build_features(test_df)

encoder = LabelEncoder()
y_train_raw = encoder.fit_transform(train_df['answer'])

print(f"Engineered Tensor (Train): {X_train_raw.shape}")
print(f"Engineered Tensor (Test):  {X_test_raw.shape}")

Engineered Tensor (Train): (2000, 5, 8)
Engineered Tensor (Test):  (500, 5, 8)


## Forward Neural Network Architecture

In [6]:
class OptionScorer(nn.Module):
    def __init__(self, in_features, hidden_dim, dropout):
        super(OptionScorer, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        B, K, F = x.shape
        return self.net(x.reshape(B * K, F)).reshape(B, K)

def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

## Training Loop with

In [7]:
num_feats = X_train_raw.shape[2]
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(X_train_raw), 5))
test_probs_sum = np.zeros((len(X_test_raw), 5))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_raw, y_train_raw)):
    Xtr, Xva = X_train_raw[tr_idx], X_train_raw[va_idx]
    ytr, yva = y_train_raw[tr_idx], y_train_raw[va_idx]

    mean = Xtr.reshape(-1, num_feats).mean(0)
    std = Xtr.reshape(-1, num_feats).std(0) + 1e-6
    Xtr_n, Xva_n = (Xtr - mean) / std, (Xva - mean) / std
    Xte_n = (X_test_raw - mean) / std

    Xtr_t = torch.tensor(Xtr_n, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(device)
    Xva_t = torch.tensor(Xva_n, dtype=torch.float32).to(device)
    Xte_t = torch.tensor(Xte_n, dtype=torch.float32).to(device)

    model = OptionScorer(num_feats, config.hidden_dim, config.dropout).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=config.batch_size, shuffle=True)

    for epoch in range(1, config.epochs + 1):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

        if epoch % 20 == 0 or epoch == config.epochs:
            model.eval()
            with torch.no_grad():
                va_probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()
            wandb.log({"fold": fold, "epoch": epoch, "val_map3": map_at_3(va_probs, yva)})

    model.eval()
    with torch.no_grad():
        va_probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()
        te_probs = torch.softmax(model(Xte_t), dim=1).cpu().numpy()

    fold_map3 = map_at_3(va_probs, yva)
    oof_probs[va_idx] = va_probs
    test_probs_sum += te_probs
    fold_scores.append(fold_map3)
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

oof_map3 = map_at_3(oof_probs, y_train_raw)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

Fold 0 | val MAP@3: 0.9179
Fold 1 | val MAP@3: 0.9300
Fold 2 | val MAP@3: 0.9242
Fold 3 | val MAP@3: 0.9187
Fold 4 | val MAP@3: 0.9317


epoch,▁▂▂▃▃▅▅▁▂▃▅▅▆▇█▂▂▃▄▅▆██▂▂▃▄▅▅▆▇██▁▂▄▅▅▆█
fold,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆████████
mean_fold_map3,▁
oof_map3,▁
val_map3,▂▁▃▅▅▄▆▅▅▆▂▅▆▆▆▆▇▃▄▄▅▅▅▅▆▂▄▄▄▄▅▃▅▆▆▆▇█▇▇
epoch,250
fold,4
mean_fold_map3,0.9245
oof_map3,0.9245
val_map3,0.93167


Mean fold MAP@3: 0.9245
Overall OOF MAP@3: 0.9245


## Rank Inference & Submission Export

In [8]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head(5))

,id,prediction
0,1,A D B
1,2,B D C
2,3,B E D
3,4,E B C
4,5,C A D
